# Module 3: From rubric to eval suite

1. The eval spreadsheet: one PASS/FAIL column per failure mode.
2. Is an improvement real? Intervals and McNemar's test.
3. **Exercise 4b (20 min):** label 15 traces, write a reference-based eval for the device classifier,
   and write two code checks.
4. Stretch goals: fix a failure in the prompt, and compare a second model.

In [ ]:
# Setup: run this cell first. It works in Google Colab and on your own laptop.
import os, sys
REPO_URL = "https://github.com/MarinaWyss/evaluating-ai-systems"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EvalsWorkshop"):
        !git clone -q {REPO_URL} /content/EvalsWorkshop
        !pip install -q litellm
    os.chdir("/content/EvalsWorkshop")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from beefcake import llm
llm.load_colab_secrets()
if llm.has_api_key():
    print("API key found. Bot model:", llm.get_model())
else:
    print("No API key found, so this notebook runs in offline mode with pre-generated data.")

## 1. The eval spreadsheet

Each row is a trace. Each failure mode is a column. Each cell is PASS or FAIL.

In [ ]:
from beefcake.evals import FAILURE_MODES, failure_rates, mcnemar_exact, run_checks, compare_to_labels
from beefcake.traces import load_traces

before = load_traces("traces_v1_labeled.csv")
before[["Trace ID", "User Query"] + FAILURE_MODES].head(8)

The bottom row of the spreadsheet is a COUNTIF per column. Here it is, with a 95% interval for each rate:

In [ ]:
failure_rates(before)

## 2. Is the improvement real?

We added one rule to the prompt (version v1.1):

In [ ]:
from beefcake.bot import PROMPTS
print(PROMPTS["v1.1"][len(PROMPTS["v1.0"]):].strip())

In [ ]:
after = load_traces("traces_v1.1_labeled.csv")
failure_rates(after)

Invents policy went from 8 of 25 to 4 of 25. The intervals overlap a lot, so compare question by question:
the only questions that tell you anything are the ones that flipped.

In [ ]:
mcnemar_exact(before["Invents policy"], after["Invents policy"])

Four questions got fixed and none broke, and it still isn't significant at the usual 0.05 level with only
25 questions. That's why you want around 100.

## 3. Exercise 4b: Build the suite (20 min)

### Step 1: label 15 traces with your rubric

This cell writes a blank eval sheet. Open `data/my_eval_sheet.csv` in a spreadsheet, fill in PASS or FAIL,
save it, and run the next cell. Change `MY_FAILURE_MODES` to the modes your group found.

In [ ]:
MY_FAILURE_MODES = FAILURE_MODES  # or your own list, for example ["Invents policy", "Assumes device"]

sheet = load_traces("traces_v1.csv")[["Trace ID", "User Query", "AI Response"]].head(15).copy()
for mode in MY_FAILURE_MODES:
    sheet[mode] = ""
sheet.to_csv("data/my_eval_sheet.csv", index=False)
print("Wrote data/my_eval_sheet.csv")

In [ ]:
mine = pd.read_csv("data/my_eval_sheet.csv", keep_default_na=False)
failure_rates(mine, MY_FAILURE_MODES)

### Step 2: a reference-based eval for the device classifier

The bot has a small classifier that says which product a question is about. There's exactly one right answer
per question, so this is a normal test. Finish `eval_device_classifier` so it returns `"PASS"` when the
prediction matches the expected label and `"FAIL"` otherwise.

Without an API key, it uses example predictions saved in the golden set.

In [ ]:
from beefcake.classifier import classify_device

golden = pd.read_csv("data/classifier_golden.csv")
EXAMPLE_PREDICTIONS = dict(zip(golden["User Query"], golden["Example prediction"]))

def eval_device_classifier(query, expected):
    predicted = classify_device(query) if llm.has_api_key() else EXAMPLE_PREDICTIONS[query]
    # YOUR CODE HERE: return "PASS" or "FAIL"
    ...

In [ ]:
golden["Result"] = [eval_device_classifier(q, e) for q, e in zip(golden["User Query"], golden["Expected"])]
print(f"Accuracy: {(golden['Result'] == 'PASS').mean():.0%}")
golden

### Step 3: two code checks

A code check is a plain function that looks at one trace and returns True if it passes. Here's one for
"Leaks RAG setup". Write another for an easy failure mode, then run both over every trace and compare
with the human labels.

Ideas: every "N-year warranty" in an answer must match the policy; the trial must be "14-day"; a return
window must be 30 days. Watch for checks that are too broad and fail good answers.

In [ ]:
def no_rag_leak(row):
    """PASS if the answer doesn't talk about 'the provided context'."""
    return "provided context" not in row["AI Response"].lower()

def my_check(row):
    # YOUR CODE HERE: return True if the trace passes
    return True

checked = run_checks(before, {"Check: no RAG leak": no_rag_leak, "Check: mine": my_check})
checked[["Trace ID", "User Query", "Check: no RAG leak", "Leaks RAG setup", "Check: mine"]]

Where does the code check disagree with the human label? An empty table means they agree on every trace.

In [ ]:
compare_to_labels(checked, "Check: no RAG leak", "Leaks RAG setup")

## 4. Stretch goals

### Fix one failure in the prompt

Add a rule to the prompt, rerun the questions that failed, and read the new answers. For a fair comparison,
regenerate the "before" answers with the same live model too, because the pre-generated traces weren't written
by your model.

In [ ]:
from beefcake.bot import answer

PROMPTS["my_fix"] = PROMPTS["v1.0"] + " YOUR RULE HERE"

failing = before[before["Invents policy"] == "FAIL"]
if llm.has_api_key():
    for _, row in failing.iterrows():
        new = answer(row["User Query"], prompt_version="my_fix")
        print(row["Trace ID"], row["User Query"], "\n  ->", new.ai_response, "\n")
else:
    print("Needs an API key.")

### Compare a second model on quality, cost, and latency

In [ ]:
OTHER_MODEL = "anthropic/claude-haiku-4-5"  # any LiteLLM model name you have a key for

if llm.has_api_key():
    rows = []
    for q in load_traces("traces_v1.csv")["User Query"].head(5):
        for model in [llm.get_model(), OTHER_MODEL]:
            t = answer(q, model=model)
            rows.append({"model": model, "question": q, "answer": t.ai_response,
                         "latency_s": llm.LAST_CALL.get("latency_s"), "cost_usd": llm.LAST_CALL.get("cost_usd")})
    comparison = pd.DataFrame(rows)
    display(comparison.groupby("model")[["latency_s", "cost_usd"]].mean())
    display(comparison)
else:
    print("Needs an API key.")

---
## Solutions (try it yourself first)

In [ ]:
def eval_device_classifier_solution(query, expected):
    predicted = classify_device(query) if llm.has_api_key() else EXAMPLE_PREDICTIONS[query]
    return "PASS" if predicted == expected else "FAIL"

import re

def warranty_matches_policy(row):
    """PASS if every 'N-year warranty' in the answer is 1 or 2 years. (Crude: it doesn't check which product.)"""
    years = re.findall(r"(\d+|one|two|three)-year warranty", row["AI Response"].lower())
    return all(y in {"1", "2", "one", "two"} for y in years)

golden["Solution"] = [eval_device_classifier_solution(q, e) for q, e in zip(golden["User Query"], golden["Expected"])]
print(f"Classifier accuracy: {(golden['Solution'] == 'PASS').mean():.0%}")
display(golden[golden["Solution"] == "FAIL"])

checked = run_checks(before, {"Check: warranty": warranty_matches_policy})
checked.loc[checked["Check: warranty"] == "FAIL", ["Trace ID", "User Query", "Invents policy"]]